In [25]:
# Preprocess Data for ML model
# Evaluating Fairness in Data-Driven Heat Vulnerability Risk Prediction
# 6.C511
# Justin Liaw, Shreeya Parekh, Julia Lukens

### Inputs:
    # HHI dataset (ZIP code level) — 25 indicator columns + Overall HHI Ranking column
    # CDC WONDER mortality data (county level) — heat-related death counts 
    # Census Bureau ZIP-to-county FIPS crosswalk

### Outputs:
# One merged dataframe at county level containing:
    # FIPS code
    # 25 population-weighted HHI indicator columns
    # Overall HHI Ranking column (population-weighted, aggregated to county)
    # Binary label column (1 = high-risk, 0 = low-risk)
    # Income quartile column (1-4)
    # Data quality column (number of ZIP codes averaged per county) (loawer priority)
    # Suppression flag column (boolean) - for counties where actual number of heat-related deaths was below 10 and CDC chose not to report it to protect privacy.

### Tasks:
    # Download HHI dataset and CDC WONDER mortality data
    # Download ZIP-to-FIPS crosswalk from Census Bureau
    # Merge crosswalk with HHI data to assign each ZIP code to a county
    # Aggregate all 25 HHI indicators and Overall HHI Ranking to county level using population-weighted averaging
    # Calculate heat-related mortality rate per county from CDC WONDER
    # Derive binary label — top quartile of mortality rate = 1, all others = 0
    # Derive income quartile column from the relevant HHI sociodemographic indicator
    # Flag counties with suppressed CDC WONDER counts
    # Add data quality column counting ZIP codes per county
    # Deliver clean merged dataframe with agreed-upon column names

In [26]:
import os
import pandas as pd
import numpy as np 
import glob

In [27]:
# Read in data

hhi = pd.read_excel("./HHI_data/HHI Data 2024 United States.xlsx", dtype={"ZCTA": str})

mortality = pd.read_csv(
    "./CDC_WONDER/Multiple Cause of Death, 1999-2020 (7).csv",
    dtype={
        "County Code": str
    }
)

xwalk = pd.read_csv(
    "./spatial_crosswalk/tab20_zcta520_county20_natl.txt", 
    sep="|",
    dtype={
        "GEOID_ZCTA5_20": str,
        "GEOID_COUNTY_20": str
    }
)

In [28]:
# Population-weighted aggregation of HHI data to county level and derivation of income quartile

# ----------------------------
# 1. Clean Columns
# ----------------------------
hhi["ZCTA"] = hhi["ZCTA"].astype(str).str.zfill(5)
xwalk["ZCTA"] = xwalk["GEOID_ZCTA5_20"].astype(str).str.zfill(5)
xwalk["county_fips"] = xwalk["GEOID_COUNTY_20"].astype(str).str.zfill(5)

# ----------------------------
# 2. Merge county crosswalk onto HHI ZCTAs
# ----------------------------
merged = hhi.merge(
    xwalk[["ZCTA", "county_fips", "NAMELSAD_COUNTY_20", "AREALAND_PART"]],
    on="ZCTA",
    how="inner"
)

# ----------------------------
# 3. Create ZCTA-to-county population weights
# ----------------------------
# We want to use population weighting to aggregate to county level.
# First, how much of each ZCTA’s population belongs to each county?
# The crosswalk does not have population,
# so approximate county-share population using land-area share.
# Assumption: population is evenly distributed within a ZCTA.
# Then use HHI POP to weight county aggregation.

# compute total area of each ZCTA
merged["zcta_area_total"] = merged.groupby("ZCTA")["AREALAND_PART"].transform("sum")  # AREALAND_PART = Calculated land area of the overlapping part in square meters

# compute area share = area in county X / total ZCTA area
merged["zcta_county_area_share"] = (merged["AREALAND_PART"] / merged["zcta_area_total"])

# allocate total ZCTA population across counties
merged["pop_allocated_to_county"] = (merged["POP"] * merged["zcta_county_area_share"])

# ----------------------------
# 4. Choose HHI columns to aggregate
# ----------------------------
# These are the HHI indicator/rank/score columns.
# Exclude IDs, names, and raw population (we'll use "pop_allocated_to_county" instead!).
exclude_cols = [
    "STATEFP10", "STATE", "STATE_ABV", "ZCTA", "GEOID10",
    "MULTI_STATE", "POP"
]

numeric_cols = hhi.select_dtypes(include=[np.number]).columns.tolist()

hhi_feature_cols = [
    c for c in numeric_cols
    if c not in exclude_cols
]

# ----------------------------
# 5. Population-weighted county averages
# ----------------------------

# fcn to calculate population-weighted average HHI indicator value for each county:
    # 1. group the merged ZCTA-county data by county
    # 2. for each county, compute population-weighted average of every HHI indicator using allocated ZCTA population as weights

def weighted_average(group, cols, weight_col):
    weights = group[weight_col] # "pop_allocated_to_county"
    out = {}

    for col in cols:
        values = group[col]
        mask = values.notna() & weights.notna() # only keep rows where both the indicator value and population weight exist

        if mask.sum() == 0 or weights[mask].sum() == 0:
            out[col] = np.nan
        else:
            out[col] = np.average(values[mask], weights=weights[mask]) # compute population-weighted average

    return pd.Series(out)

# apply fcn to each county
county_hhi = (
    merged
    .groupby(["county_fips", "NAMELSAD_COUNTY_20"])
    .apply(
        weighted_average,
        cols=hhi_feature_cols,
        weight_col="pop_allocated_to_county"
    )
    .reset_index()
)

# ----------------------------
# 6. Add total allocated population per county from all contributing ZCTAs
# ----------------------------
county_pop = (
    merged
    .groupby(["county_fips", "NAMELSAD_COUNTY_20"])["pop_allocated_to_county"]
    .sum()
    .reset_index()
    .rename(columns={"pop_allocated_to_county": "county_pop_from_zctas"})
)

county_hhi = county_hhi.merge(
    county_pop,
    on=["county_fips", "NAMELSAD_COUNTY_20"],
    how="left"
)

# ----------------------------
# 7. Add a data quality column counting ZIP codes per county (overlap)
# ----------------------------
county_zip_counts = (
    merged
    .groupby(["county_fips", "NAMELSAD_COUNTY_20"])["ZCTA"]
    .nunique()
    .reset_index()
    .rename(columns={"ZCTA": "n_zctas_in_county"})
)

county_hhi = county_hhi.merge(
    county_zip_counts,
    on=["county_fips", "NAMELSAD_COUNTY_20"],
    how="left"
)

# ----------------------------
# 8. Derive poverty quartile
# ----------------------------
# 1 = bottom 25%
# 4 = top 25%

# TODO recommend: 
# quartiles → exploratory analysis
# binary high_poverty → Fairlearn metrics
# because many fairness metrics work more cleanly with two groups

county_hhi["poverty_quartile"] = pd.qcut(
    county_hhi["PR_POV"],
    q=4,
    labels=[1, 2, 3, 4]
)

county_hhi["poverty_quartile"] = (
    county_hhi["poverty_quartile"]
    .astype(int)
)

# ----------------------------
# 9. Save output
# ----------------------------
county_hhi.to_csv("county_level_hhi_population_weighted.csv", index=False)

print(county_hhi.shape)
print(county_hhi.head())



(3108, 73)
  county_fips NAMELSAD_COUNTY_20     PR_HRI     F_HRI   LOW_EMS      P_NEHD  \
0       01001     Autauga County  64.365804  0.756453  0.000000  -18.438789   
1       01003     Baldwin County  75.842151  0.297279  0.002448  -29.605982   
2       01005     Barbour County  56.780091  0.722064  0.001760    9.767859   
3       01007        Bibb County  72.725062  0.839975  0.004131 -216.191705   
4       01009      Blount County  59.495221 -1.335081 -2.047257  -57.887124   

      PR_NEHD  HHB_SCORE  HHB_RANK     P_CHD  ...  PR_OZONE    P_PM25  \
0  -26.215642   0.479537  0.372070  5.978737  ...  0.377930  0.000000   
1  -38.983087   0.595870  0.566914  6.719081  ...  0.217727  0.000000   
2    0.270125   0.496047  0.390836  8.424020  ...  0.000000  0.000000   
3 -221.100070   0.559280  0.497766  6.775043  ...  0.528481  0.000000   
4  -61.469182  -1.639118 -1.791044  7.291256  ...  0.295537  0.004112   

    PR_PM25  NBE_SCORE  NBE_RANK  OVERALL_SCORE  OVERALL_RANK  \
0  0.00000

In [29]:
# Derive mortality data label

# Clean up columns
mortality["county_fips"] = mortality["County Code"].astype(str).str.zfill(5)

#print(mortality["Deaths"].unique())

# Add column replacing suppressed data with the midpoint of possible suppressed values
mortality["Deaths_adjusted"] = (
    mortality["Deaths"]
    .replace("Suppressed", 5).astype(int)
)

# Compute crude rate adjusted 
mortality["Crude Rate_adjusted"] = (
    mortality["Deaths_adjusted"] / mortality["Population"]
)

# Create label
threshold = mortality["Crude Rate_adjusted"].quantile(0.75)
mortality["label"] = (mortality["Crude Rate_adjusted"] >= threshold).astype(int) # 1 = high risk (top 25%)

# save mortality rates with labels to file
mortality.to_csv("./CDC_WONDER/mortality_data_labels.csv", index=False)

In [30]:
# Merge label onto HHI county-level dataset

county_hhi = county_hhi.merge(
    mortality,
    on=["county_fips"],
    how="left"
)

# save mortality rates with labels to file
county_hhi.to_csv("./heat_risk_dataframe.csv", index=False)